# ✈️ Unlocking Behavioral Intelligence in Airline Loyalty Programs
**Lead Strategist:** Krrish Kumar  
**Evaluation:** IIT Guwahati Consulting & Analytics Club

---

### 🏛️ Executive Summary
Standard Customer Lifetime Value (CLV) models fail to capture latent churn—high-value flyers who silently disengage without formally canceling their loyalty cards. This architecture deploys a **Strict Temporal Firewall**, analyzing historical behavior to predict future disengagement without data leakage.

### ⚙️ Core Pipeline Architecture
1. **Dynamic Churn Definition:** Identifies "Silent Attrition" (Highly active historically, zero activity in the target window) rather than relying solely on formal card cancellations.
2. **Temporal Firewall:** Splits data strictly by year to ensure the XGBoost model cannot "look into the future."
3. **Behavioral Segmentation:** K-Means clustering to identify distinct cohorts beyond basic CLV (e.g., Elite Road Warriors vs. Seasonal Deal Hunters).
4. **Actionable Retention Engine:** Exports a scored customer matrix to power the Executive Digital Twin dashboard.

In [2]:
# ================================================================
# 01. INGESTION & STRICT TEMPORAL FIREWALL (LOCAL EXECUTION)
# ================================================================
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
np.random.seed(42)

print("🚀 Initializing Airline Loyalty Pipeline (Local Mode)...")

path_history = 'airline_loyalty_data/Customer Loyalty History.csv'
path_activity = 'airline_loyalty_data/Customer Flight Activity.csv'

try:
    df_history = pd.read_csv(path_history)
    df_activity = pd.read_csv(path_activity)
    print(f"✅ Local Datasets Loaded. Tracking {len(df_history):,} members.")
except FileNotFoundError:
    print(f"❌ CRITICAL ERROR: Ensure datasets are inside 'airline_loyalty_data/'.")
    raise

# --- THE TEMPORAL FIREWALL ---
# We train the model to understand what 2017 behavior leads to 2018 churn.
df_2017 = df_activity[df_activity['Year'] == 2017]
df_2018 = df_activity[df_activity['Year'] == 2018]

# Engineer 2017 Behavioral Signals (The Predictors)
behavior_2017 = df_2017.groupby('Loyalty Number').agg(
    flights_2017=('Flights Booked', 'sum'),
    distance_2017=('Distance', 'sum'),
    points_earned_2017=('Points Accumulated', 'sum'),
    points_redeemed_2017=('Points Redeemed', 'sum')
).reset_index()

# Define "Silent Churn" (The Target)
# Definition: Flew in 2017, but completely stopped flying in 2018
behavior_2018 = df_2018.groupby('Loyalty Number').agg(flights_2018=('Flights Booked', 'sum')).reset_index()

df_master = pd.merge(df_history, behavior_2017, on='Loyalty Number', how='inner')
df_master = pd.merge(df_master, behavior_2018, on='Loyalty Number', how='left')
df_master['flights_2018'] = df_master['flights_2018'].fillna(0)

# Target Variable: 1 if Churned (0 flights in 2018), 0 if Retained
df_master['Silent_Churn_Flag'] = np.where((df_master['flights_2017'] > 0) & (df_master['flights_2018'] == 0), 1, 0)

# Isolate to only members who were active in 2017 (our addressable market)
df_master = df_master[df_master['flights_2017'] > 0].reset_index(drop=True)

# Clean demographics
df_master['Salary'] = pd.to_numeric(df_master['Salary'], errors='coerce')
df_master['Salary'] = df_master['Salary'].fillna(df_master['Salary'].median())

print(f"📊 Temporal Firewall Active. Analyzable Active Base: {len(df_master):,} members.")
print(f"⚠️ Baseline Silent Churn Rate: {df_master['Silent_Churn_Flag'].mean()*100:.2f}%")

🚀 Initializing Airline Loyalty Pipeline (Local Mode)...
✅ Local Datasets Loaded. Tracking 16,737 members.
📊 Temporal Firewall Active. Analyzable Active Base: 12,836 members.
⚠️ Baseline Silent Churn Rate: 4.37%


In [3]:
# ================================================================
# 02. SEGMENTATION & XGBOOST EARLY WARNING SYSTEM
# ================================================================
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Behavioral Value Segmentation (K-Means)
seg_features = df_master[['flights_2017', 'distance_2017', 'CLV']]
scaler = StandardScaler()
seg_scaled = scaler.fit_transform(seg_features)

kmeans = KMeans(n_clusters=3, random_state=42)
df_master['Cluster'] = kmeans.fit_predict(seg_scaled)

cluster_mapping = {
    0: 'Occasional Leisure',
    1: 'Elite Road Warriors',
    2: 'High-Value Seasonal'
}
df_master['Strategic_Segment'] = df_master['Cluster'].map(cluster_mapping)

# 2. XGBoost Early Warning Engine
features = ['flights_2017', 'distance_2017', 'points_redeemed_2017', 'CLV', 'Salary']
cat_cols = ['Gender', 'Education', 'Marital Status', 'Loyalty Card']

df_ml = pd.get_dummies(df_master, columns=cat_cols, drop_first=True)
valid_features = [c for c in df_ml.columns if c in features or any(cat in c for cat in cat_cols)]

X = df_ml[valid_features]
y = df_ml['Silent_Churn_Flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Handle class imbalance natively
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model = XGBClassifier(scale_pos_weight=scale_weight, random_state=42, max_depth=4, learning_rate=0.05)
xgb_model.fit(X_train, y_train)

# 3. Model Validation
preds = xgb_model.predict_proba(X_test)[:, 1]
print(f"🏆 XGBoost Early Warning Engine Active | Out-of-Sample ROC-AUC: {roc_auc_score(y_test, preds):.4f}")

# 4. Score the entire active base for the Marketing Dashboard
df_master['Churn_Risk_Score'] = xgb_model.predict_proba(X)[:, 1]

# Export the scored data to power the Streamlit Digital Twin
export_path = 'airline_loyalty_data/scored_members.csv'
df_master.to_csv(export_path, index=False)
print(f"✅ Intelligence Matrix successfully exported to '{export_path}' for Dashboard consumption.")

🏆 XGBoost Early Warning Engine Active | Out-of-Sample ROC-AUC: 0.7930
✅ Intelligence Matrix successfully exported to 'airline_loyalty_data/scored_members.csv' for Dashboard consumption.
